### THOUTHS ON RUNNING IIZUKA MODEL

### MASKS

We consider different types of masks generated by varying the corruption **method**, **ratio**, and optional parameters such as **looping** and **border constraints**.

## 1. Mask Types (method)

We consider five possible mask generation methods:

* **SAP (salt-and-pepper noise)**
* **Box**
* **Circle**
* **Random Walk (rw)**
* **All (mixed random-walk regime)**

---

## SAP (Salt-and-Pepper Noise)

* **Characteristics:**
  * Sparse, unstructured corruption
  * Pixels are independent
  * No spatial continuity
* **What it tests:**
  * Local texture reconstruction
  * Denoising capability
* **Not suitable for:**
  * Semantic or structural reasoning
* **Expected difficulty:**
  * Usually the easiest case
* **Possible variations:**
  * Only the corruption ratio

---

## Box Masks

* **Characteristics:**
  * Large contiguous missing regions
  * Simple geometric structure
* **What it tests:**
  * Structural completion
  * Semantic inpainting of missing objects or regions
* **Notes:**
  * Often requires object-level reasoning
* **Possible variations:**
  * Corruption ratio (controls missing area)

---

## Circle Masks

* **Characteristics:**
  * Similar to box masks in area coverage
  * Smooth boundary geometry
* **What it tests:**
  * Sensitivity to boundary shape (sharp vs smooth edges)
  * Structural completion
* **Key insight:**
  * Comparing box vs circle (same area) helps evaluate whether **boundary geometry affects performance**
* **Possible variations:**
  * Corruption ratio

---

## Random Walk (rw) Masks

* **Characteristics:**
  * Thin, irregular, and connected structures
  * Highly non-convex shapes
* **What it tests:**
  * Long-range contextual reasoning
  * Robustness to irregular missing regions
  * Realistic corruption patterns (e.g., scratches, object removal artifacts)
* **Difficulty:**
  * Often harder than box/circle masks due to irregular connectivity
* **Possible variations:**
  * Corruption ratio

---

## Mixed (“all” mode)

* **Description:**
  * Random-walk masks with varying predefined ratios
* **Note:**
  * Not recommended for controlled evaluation
* **Reason:**
  * Reduces interpretability of comparisons between mask types

---

## 2. Corruption Ratio

A reasonable set of values for evaluation is:

* **0.1**
* **0.2**
* **0.3**
* **0.4**

These values allow analysis of performance degradation as the missing area increases.

---

## 3. Looping Parameter

* Not particularly useful for evaluation
* It introduces artificial continuity effects that are not representative of real corruption patterns

---

## 4. Border Constraint

* Not useful for analysis
* Can bias masks toward center or exclude edge regions in an artificial way

---

# Summary

For a clean and interpretable experimental study, the most meaningful factors are:

* **Mask type:** SAP, Box, Circle, Random Walk
* **Corruption ratio:** 0.1, 0.25, 0.4

# Final masks variety for iizuka
* 1-3: sap, ratio 0.1 (too sparse pixel to differentiate center-border)
* 4-6: sap, ratio 0.25 (too sparse pixel to differentiate center-border)
* 7-9: sap, ratio 0.4 (too sparse pixel to differentiate center-border)
* 10-12: box, ratio 0.1, centered
* 13-15: box, ratio 0.1, bordered
* 16-18: box, ratio 0.25, centered
* 19-21: box, ratio 0.25, bordered
* 22: box, ratio 0.4, centered
* 23-24: box, ratio 0.4, bordered
* 25-27: circle, ratio 0.1, centered
* 28-30: circle, ratio 0.1, bordered
* 31-33: circle, ratio 0.25, centered
* 34-36: circle, ratio 0.25, bordered
* 37: circle, ratio 0.4, centered
* 38: circle, ratio 0.4, bordered
* 39-40: rw, ratio 0.1, centered
* 41-42: rw, ratio 0.1, bordered
* 43-44: rw, ratio 0.25, centered
* 45-46: rw, ratio 0.25, bordered
* 47-48: rw, ratio 0.4 (too sparse pixel to differentiate center-border)

## CONCLUSIONI VARIE SU IIZUKA

Confronto blending vs no blending, a occhio:
* sap: blending riesce a pulire un po' meglio. Con alto ratio però, si formano strani artifici grafici come di puntini rimasti allineati (img 06_09)
* bordered: se i bordi mancanti sono spessi, poisson blending sballa completamente peggiorando la situazione mettendo tutto bianco nel patch (vedi generata 01_020). Nei bordi piccoli può migliorare di poco il blending del bordo di congiunzione tra patch area e gt area (vedi generata 01_13)
* centered: con piccoli patch centrali di poco meglio con blending. Con patch centrali piu grossi, quando il colore da iizuka base viene piu modificato, il blending aiuta di più (vedi imm 01_25, 05_26). caso di peggioramento per qualche motivo genrata 05_011.
* rw: blending riesce a sistemare la gamma colori dissimulando l'artificio grafico creato dal patch (vedi generata 01_041, 05_39). rimane un problema se rw tocca il bordo (vedi generata 01_045)

Generale:
* mi sembra funzioni meglio su box che su cerchi, in particolare iizuka standard senza blending
* fallisce quando mancano grosse informazioni semantiche, tipo se una grossa parte di un oggetto è oscurata (img 05_017). invece casi di completamento di un oggetto quando ne è presente solo una parte: 115_017
* colorazione strana (05_043, 06_41,07_014, 125_24, 146_15), sarebbe bello capir che succede 
* allungamento di oggetto img 06_021, 122_11
* casi di object removal involontaria (quando la maschera coincide piu o meno con un oggetto) (in questo caso le metriche daranno punteggi pessimi essendo che confrontano immagine generata con ground truth) (img 08_22, 104_18, 122_18)
* casi di creazione di oggetti, (img 105_017 forse sono due facce)
* quando il patch è grande, wquello generato puo essere molto diverso dall'orginale pur rimanendo verosimile (img 112_22)
* in casi in cui è rappresentata una struttura ordinata l'occhio umano riconosce piu facilmente l'artificio grafico (img 106_16). In casi di elementi non ordinati è molto più ingannabile (img 111_017)
* ovvi problemi di ricostruzione delle scritte (img 119_015)
* buoni risultati anche per patch grandi se l'immagine è già abbastanza uniforme per conto suo (img 125_3, 125_37)

## CONCLUSIONI PER METRICS ANALYSIS
Comments on metrics analysis (standard iizuka vs blending version):

COMPARISON:
1) plot_metric_histograms: L1, PSNR e SSIM sono visibilmente migliorate come valori; però in L1 e ancor di più in PSNR abbiamo una coda di valori peggiorati (vedi plot_mask_difficulty); LPIPS rimane simile
2) plot_heatmap: su L1 e PSNR c'è un visibile miglioramento per le sap e rw masks, un peggioramento per le border masks; SSIM e LPIPS rimangono simili. Presenza di immagini ostiche per L1. vedi anche plot_method_comparison
3) plot_mask_difficulty: L1 e PSNR migliorano su sap e rw, peggiorano su bordered; SSIM e LPIPS migliorano su sap e rw, sul resto rimangono simili; vedi anche plot_method_comparison
4) plot_image_difficulty: miglioramento più o meno uniforme (su tutte le immagini) di SSIM e LPIPS, mentre L1 e PSNR rimangono simili
5) plot_metric_vs_ratio: per L1 e PSNR i risultati peggiorano circa linearmente con l'aumentare del ratio di pixel mascherati (come da aspettativa): qui in standard iizuka i peggiori sono le circles mask (peggio delle box), mentre in blending sono i bordered; per SSIM e LPIPS i peggiori in modo evidente rispetto agli altri sono i sap per entrambi i metodi


IN GENERALE:
1) find_extreme_cases: in standard, immagine peggiore per L1 e PSNR è 21_38; in blending immagine peggiore L1 e PSNR è 41_23; sia per iizuka standard che per blending cas, la peggiore per SSIM è 183_07, per LPIPS è 161_07;  
